# Playfit Intelligence Lab — 03: Hybrid Recommender Model

Sistema de recomendación híbrido: content-based + popularity priors + penalización por calidad de datos.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import polars as pl

from src.features.game_features import build_feature_matrix, compute_popularity_score, compute_richness_score
from src.models.content_based import build_content_model
from src.models.hybrid import HybridRecommender
from src.evaluation.explainability import make_explanation, get_game_details

print("Librerías y módulos cargados.")

## 1. Preparación del Modelo
Cargamos la feature matrix y construimos el modelo content-based.

In [ ]:
fm = build_feature_matrix()
fm = compute_popularity_score(fm)
fm = compute_richness_score(fm)
print(f"Feature matrix: {fm.shape[0]:,} × {fm.shape[1]}")

cm = build_content_model(fm, n_components=100)
print(f"Content model construido (SVD-100)")

## 2. Hybrid Recommender

`final_score = α × content_score + β × popularity_score — γ × confidence_penalty`

In [ ]:
rec = HybridRecommender(alpha=0.5, beta=0.4, gamma=0.1)
rec.fit(fm, cm)
print("Modelo híbrido listo.")

## 3. Ejemplo: Recomendar para juegos conocidos

In [ ]:
liked = ['the_legend_of_zelda_breath_of_the_wild', 'super_mario_odyssey']
results = rec.recommend(liked, k=5)

print(f"Basado en: {liked}")
print("=" * 80)
for r in results:
    details = get_game_details(r['game_id'])
    expl = make_explanation(r)
    print(f"  {r['game_id']}")
    print(f"    Score: {r['final_score']:.3f} | Content: {r['content_score']:.3f} | Pop: {r['popularity_score']:.3f} | Conf: {r['data_confidence']}")
    print(f"    → {expl}")
    print()

## 4. Cold-Start: Nuevo usuario sin historial

In [ ]:
cold_results = rec.recommend([], k=5)
print("Cold-start (popularidad global):")
for r in cold_results:
    print(f"  {r['game_id']}: score={r['final_score']:.2f}, conf={r['data_confidence']}")

## 5. Recomendación por Perfil de Tags

In [ ]:
profile_recs = rec.recommend_for_profile(['story_rich', 'atmospheric', 'exploration'], k=5)
print("Para perfil: story_rich + atmospheric + exploration")
for r in profile_recs:
    print(f"  {r['game_id']}: score={r['final_score']:.2f}")

## 6. Experimentación con Hiperparámetros (α, β, γ)

In [ ]:
import itertools

param_grid = [
    (0.6, 0.3, 0.1), (0.5, 0.4, 0.1), (0.4, 0.5, 0.1),
    (0.7, 0.2, 0.1), (0.5, 0.3, 0.2), (0.4, 0.4, 0.2),
]

test_liked = ['the_legend_of_zelda_breath_of_the_wild', 'hollow_knight', 'portal_2']
test_excluded = []

print(f"{'α':>5} {'β':>5} {'γ':>5}  {'Top-1':>40} {'Score':>8}")
print("-" * 65)

results_summary = []
for alpha, beta, gamma in param_grid:
    r = HybridRecommender(alpha=alpha, beta=beta, gamma=gamma)
    r.fit(fm, cm)
    recs = r.recommend(test_liked, k=3)
    top = recs[0]['game_id'] if recs else 'N/A'
    score = recs[0]['final_score'] if recs else 0.0
    print(f"{alpha:5.1f} {beta:5.1f} {gamma:5.1f}  {top:>40} {score:8.3f}")
    results_summary.append({'alpha': alpha, 'beta': beta, 'gamma': gamma, 'top': top, 'score': score})

print("\nLos mejores parámetros variarán según el perfil de usuario. En la práctica, se optimizarían con validación cruzada sobre datos de interacción reales.")

## 7. Resumen del Modelo

**Componentes:**
- **Content-Based (α):** Similaridad coseno sobre embedding SVD-100 de tags y plataformas
- **Popularity Prior (β):** Ranking compuesto de scores, ventas y sentimiento
- **Confidence Penalty (γ):** Penalización por baja calidad de datos

**Casos borde manejados:**
- **Cold-start:** Ranking por popularidad global cuando no hay historial
- **Datos incompletos:** Imputación de scores/ventas faltantes por mediana
- **Diversidad:** Embedding SVD captura relaciones semánticas entre juegos